# Taller de Python · Sesión 3
## Combinar tablas y auditar a la IA

**Business Analytics I: Fundamentos y herramientas** · MBDA&IA 2026-27

Al terminar este notebook sabrás:
1. Unir tablas con `merge` sin inflar las cifras.
2. Elegir con qué filas te quedas (`how`).
3. Apilar tablas con `concat` y comprobar el resultado.
4. Encontrar los errores de un análisis hecho por IA que funciona… pero miente.

Después, en `S3_proyecto_plantilla.ipynb`, harás tu propio informe.

---
## 📋 El encargo

> **Superstore · Dirección Comercial** · Nota interna
>
> Te paso **cuatro años de pedidos** (2015–2018). Vienen en **tres tablas**: pedidos, devoluciones y responsables de cada región. Queremos saber qué región **devuelve más**, quién es su responsable y si alguna región **pierde dinero**. El becario ya ha hecho un primer análisis; revísalo antes de que llegue al comité.
>
> *Dirección Comercial*

Superstore es una empresa ficticia: son los datos de ejemplo de Tableau.

## 🤖 La IA hoy: copiloto con auditoría

- **§1–§5, a mano.** Intenta escribir tú el código. Usa el Chat solo para que te **explique** un error. Así tendrás criterio para la §6 y para tu proyecto.
- **§6 y proyecto: copiloto.** La IA puede escribir código, pero **tú decides las preguntas, compruebas cada cifra y escribes las conclusiones**.
- **Si no puedes explicar una línea, no la entregas.**
- Al final hay una **bitácora de IA**: qué pediste, qué te devolvió, qué comprobaste y qué cambiaste.

---
# 0. Las tres tablas

Los datos de una empresa casi nunca vienen en una sola tabla. La celda siguiente busca la carpeta `datos` del kit; si no la encuentra (por ejemplo, en Colab), los lee del repositorio del taller.

In [ ]:
from pathlib import Path
import pandas as pd

# Busca la carpeta datos del kit; si no existe (Colab), usa el repositorio. No hace falta entender esta línea.
DATOS = "../datos" if Path("../datos").exists() else "https://raw.githubusercontent.com/javialani/taller-python-mbda-2026/main/sesion3/datos"

orders = pd.read_csv(f"{DATOS}/orders.csv")
returns = pd.read_csv(f"{DATOS}/returns.csv")
people = pd.read_csv(f"{DATOS}/people.csv")

print("orders :", orders.shape)
print("returns:", returns.shape)
print("people :", people.shape)

In [ ]:
orders[["Order ID", "Order Date", "Region", "Product Name", "Sales", "Profit"]].head()

In [ ]:
returns.head()

In [ ]:
people

Fíjate en qué cuenta cada tabla:
- **orders**: una fila por **línea de pedido**. Un pedido con tres productos ocupa tres filas.
- **returns**: una fila por **línea devuelta**, pero solo con el número de pedido.
- **people**: una responsable por región.

🟢 **Ejercicio 0.1** · ¿Qué columna une `orders` con `people`? ¿Y `orders` con `returns`? Responde en la celda de texto.

✍️ **Tu respuesta:**

- orders ↔ people: 
- orders ↔ returns: 

---
# 1. `merge`: emparejar por una columna común

`merge` es como **emparejar cada ticket con su ficha por el número que comparten**. Cada línea de pedido busca a la responsable de su región.

> **Regla de oro:** cuenta las filas **antes y después** de cada merge.

In [ ]:
pedidos = orders.merge(people, on="Region", how="left", validate="m:1")

print("Antes  :", len(orders))
print("Después:", len(pedidos))      # ¿cuántas esperabas?
pedidos[["Order ID", "Region", "Person"]].head()

- `on="Region"`: la columna por la que se empareja.
- `how="left"`: me quedo con **todas** las filas de la izquierda (`orders`). Lo vemos en la §3.
- `validate="m:1"`: «muchos a uno». Pandas comprueba que en la derecha (`people`) cada región aparece **una sola vez**; si no, da error en lugar de inflar las cifras.

🟢 **Ejercicio 1.1** · ¿Cuánto ha vendido cada responsable? Ordénalo de mayor a menor.

In [ ]:
# TU CÓDIGO AQUÍ

---
# 2. La trampa: el merge que infla las ventas

Ahora queremos saber, para cada línea, si su pedido fue devuelto. Prueba el merge «directo»:

In [ ]:
inflado = orders.merge(returns, on="Order ID", how="left")

print("Filas antes  :", len(orders))
print("Filas después:", len(inflado))
print("Ventas antes  :", round(orders["Sales"].sum()), "$")
print("Ventas después:", round(inflado["Sales"].sum()), "$")

Hay **más filas** que antes y las ventas han subido un 26 % sin vender nada más. ¿Por qué?

### 🔍 Ejercicio: ¿por qué crece?

Vamos a mirar un pedido real, `CA-2015-109918`, en cada tabla.

🟢 **Ejercicio 2.1** · Filtra ese pedido en `orders`. ¿Cuántas filas tiene? ¿Qué productos son?

In [ ]:
# TU CÓDIGO AQUÍ

🟢 **Ejercicio 2.2** · Filtra el mismo pedido en `returns`. ¿Cuántas filas tiene? ¿Dicen qué producto se devolvió?

In [ ]:
# TU CÓDIGO AQUÍ

🟢 **Ejercicio 2.3** · **Predice** cuántas filas tendrá este pedido después del merge directo. Después compruébalo filtrando `inflado`.

✍️ Mi predicción: ___ filas

In [ ]:
# TU CÓDIGO AQUÍ

🟢 **Ejercicio 2.4** · ¿Qué pedidos aparecen más veces en `returns`? Usa `value_counts()`.

In [ ]:
# TU CÓDIGO AQUÍ

**Lo que pasa:** `merge` empareja **cada fila de la izquierda con todas las filas de la derecha que tienen la misma clave**. Returns repite el pedido una vez por línea devuelta, así que cada línea del pedido se copia tantas veces como aparezca en Returns.

🔴 **Reto 2.5** · Explica las 12.420 filas. Pista: un pedido devuelto con *n* líneas genera *n* × *n* filas; las líneas no devueltas se quedan igual.

In [ ]:
# TU CÓDIGO AQUÍ

### ✅ La solución: una fila por pedido en la tabla de la derecha

Para la pregunta «¿se devolvió este pedido?» basta con **una fila por pedido**.

In [ ]:
devueltos = returns.drop_duplicates("Order ID")
print("Pedidos devueltos:", len(devueltos))

pedidos = pedidos.merge(devueltos, on="Order ID", how="left", validate="m:1")
print("Filas:", len(pedidos))                         # 9994 ✓
print("Ventas:", round(pedidos["Sales"].sum()), "$")      # sin inflar ✓

🤖 **Momento IA** · Quita el `#` de la línea siguiente y ejecuta: es el merge directo, pero con `validate="m:1"`. Copia el error en el Chat y pídele «explícame este error». Anótalo en la bitácora.

In [ ]:
# orders.merge(returns, on="Order ID", how="left", validate="m:1")

---
# 3. `how`: ¿con qué filas me quedo?

| `how` | Qué se queda |
|---|---|
| `"inner"` (**por defecto**) | Solo las filas cuya clave está en las dos tablas |
| `"left"` | Todas las de la izquierda; si no hay pareja, `NaN` |
| `"right"` | Todas las de la derecha |
| `"outer"` | Todas las de las dos |
| `"left_anti"` | Las de la izquierda **sin** pareja en la derecha |

⚠️ Si no escribes `how`, pandas usa `inner` y **pierde las filas sin pareja sin avisar**.

In [ ]:
for h in ["left", "inner", "outer", "left_anti"]:
    r = orders.merge(devueltos, on="Order ID", how=h)
    print(h, "→", len(r), "filas")

🟢 **Ejercicio 3.1** · ¿Por qué `inner` da 800 filas? ¿Te serviría para calcular el **porcentaje** de líneas devueltas?

✍️ **Tu respuesta:** 

---
# 4. ¿Qué región devuelve más?

`Returned` vale `"Yes"` si la línea se devolvió y `NaN` si no. `notna()` lo convierte en `True`/`False`, y **la media de `True`/`False` es una proporción** (como en la sesión 2).

In [ ]:
pedidos["devuelto"] = pedidos["Returned"].notna()
pedidos["devuelto"].mean()

🟢 **Ejercicio 4.1** · Tasa de devolución (por líneas) y número de líneas por **región y responsable**. Ordena de mayor a menor tasa.

In [ ]:
# TU CÓDIGO AQUÍ

🔴 **Reto 4.2** · ¿Cambia la conclusión si cuentas **pedidos** en lugar de líneas? Pista: `drop_duplicates("Order ID")`.

In [ ]:
# TU CÓDIGO AQUÍ

⭐ **Pregunta surgida de los datos** · West devuelve el 15 % de sus líneas; el resto, entre el 4 y el 5 %. Nadie lo había preguntado al empezar. ¿Será porque West vende otros productos?

🔴 **Reto 4.3** · Calcula la tasa de devolución por **categoría y región** (`pivot_table`). ¿Se explica por el tipo de producto?

In [ ]:
# TU CÓDIGO AQUÍ

---
# 5. `concat`: apilar los archivos de cada año

`merge` añade **columnas** emparejando por una clave. `concat` añade **filas**: apila tablas con las mismas columnas, como **juntar los archivadores de cada año en uno**.

Sistemas exporta un archivo por año:

In [ ]:
o2015 = pd.read_csv(f"{DATOS}/orders_2015.csv")
o2016 = pd.read_csv(f"{DATOS}/orders_2016.csv")
o2017 = pd.read_csv(f"{DATOS}/orders_2017.csv")
o2018 = pd.read_csv(f"{DATOS}/orders_2018.csv")

for nombre, t in [("2015", o2015), ("2016", o2016), ("2017", o2017), ("2018", o2018)]:
    print(nombre, t.shape)

🟢 **Ejercicio 5.1** · Apila las cuatro tablas en `todos` con `pd.concat` y `ignore_index=True`. Comprueba el número de filas: ¿es 9.994?

In [ ]:
# TU CÓDIGO AQUÍ

`ignore_index=True` renumera las filas de 0 al final. Sin él, cada tabla conserva su numeración y habría cuatro filas con el índice 0.

🟢 **Ejercicio 5.2** · Las filas cuadran. Ahora revisa las **columnas** y la **suma de ventas**. ¿Algo raro?

In [ ]:
# TU CÓDIGO AQUÍ

🟢 **Ejercicio 5.3** · Encuentra qué columna no coincide entre 2018 y 2015, corrígela con `rename` y vuelve a apilar. Comprueba filas, columnas y ventas.

In [ ]:
# TU CÓDIGO AQUÍ

> **Regla:** después de un `concat`, cuenta las filas **y revisa las columnas**. Con `join="outer"` (el valor por defecto), las columnas que no coinciden no dan error: se rellenan con `NaN`.

---
# 6. 🕵️ Auditar al becario

El becario ha dejado este análisis hecho. **El código funciona y las conclusiones suenan bien.** Hay **tres errores** y una celda correcta.

Para cada celda: ejecútala, lee la conclusión y decide si es correcta. Si no lo es, escribe por qué y corrígela debajo.

> **Pista:** compara cada cifra con algo que ya sabes (9.994 líneas, 2,30 M$ de ventas).

In [ ]:
# 🤖 Celda 1 del becario
orders = pd.read_csv(f"{DATOS}/orders.csv")       # recargamos los datos originales
returns = pd.read_csv(f"{DATOS}/returns.csv")

ventas = orders.merge(returns, on="Order ID", how="left")
print("Ventas del periodo:", round(ventas["Sales"].sum(), 2), "$")
# Conclusión del becario: «Las ventas del periodo ascienden a 2,9 millones de dólares.»

✍️ **¿Correcta? ¿Por qué?** 

In [ ]:
# Tu corrección (si hace falta)

In [ ]:
# 🤖 Celda 2 del becario
limpio = orders.dropna()
print(len(limpio), "filas")
# Conclusión del becario: «Datos limpios: eliminadas las filas incompletas.»

✍️ **¿Correcta? ¿Por qué?** 

In [ ]:
# Tu corrección (si hace falta)

In [ ]:
# 🤖 Celda 3 del becario
orders.groupby("Category")["Sales"].sum()
# Conclusión del becario: «Technology es la categoría que más vende.»

✍️ **¿Correcta? ¿Por qué?** 

In [ ]:
# Tu comprobación

In [ ]:
# 🤖 Celda 4 del becario
orders["margen"] = orders["Profit"] / orders["Sales"]
orders.groupby("Region")["margen"].mean().round(3)
# Conclusión del becario: «Central pierde un 10 % en cada venta. Recomendamos cerrarla.»

✍️ **¿Correcta? ¿Por qué?** 

In [ ]:
# Tu corrección (si hace falta)

🔴 **Reto 6.1** · ¿Por qué el margen medio de Central sale tan negativo? Mira el descuento medio por región y las líneas de Central con peor margen.

In [ ]:
# TU CÓDIGO AQUÍ

> El código que da error es fácil de arreglar. **El peligroso es el que funciona y miente.**

---
## Resumen de hoy
| Concepto | Símil o idea clave |
|---|---|
| `merge` | Emparejar ticket y ficha por un número común |
| `how="left"` / `"inner"` | Me quedo con todo / solo con lo que empareja |
| `validate="m:1"` | La comprobación queda escrita en el código |
| Contar filas | Antes y después de cada merge |
| `concat` | Apilar los archivadores de cada año |
| Revisar columnas | Después de cada concat |
| Media de ratios | Una venta de 5 $ no pesa como una de 5.000 $ |
| IA | Becario brillante: revisa todo lo que entrega |

## 📓 Bitácora de IA
| Qué pedí | Qué devolvió | Qué comprobé | Qué cambié |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

## ✅ Autoevaluación
1. ¿Qué diferencia hay entre `merge` y `concat`?
2. Tras un merge tengo más filas que antes. ¿Qué ha pasado?
3. ¿Para qué sirve `validate="m:1"`?
4. ¿Por qué no hay que hacer `dropna()` sin mirar?
5. ¿Cómo se calcula bien el margen de una región?
6. El código del becario funciona. ¿Ya está?

**Siguiente paso:** abre `S3_proyecto_plantilla.ipynb` y empieza tu proyecto.